# Coding Assignment: Lung Cancer Prediction using Machine Learning

Dataset: Kaggle Lung Cancer Dataset  
Models: Logistic Regression and Support Vector Machine (SVM)

**ABDUL KHALED ARAFAT**

**ID: 2671507**


## Phase A: Data Preprocessing

This block contains these:
Dataset download,
CSV loading,
Data inspection,
Missing-value check,
Duplicate check/removal,
Column cleaning,
Target identification,
Train/test split,
Feature selection,
Data scaling


In [ ]:
# PHASE A: DATA ENGINEERING & FEATURE SELECTION

!pip -q install kagglehub

# Step 1: Import libraries
import pandas as pd
import kagglehub
import os
import glob

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

# Step 2: Download dataset from Kaggle

path = kagglehub.dataset_download(
    "akashnath29/lung-cancer-dataset"
)

print("Dataset downloaded to:")
print(path)

# Step 3: Find the CSV file

csv_files = glob.glob(
    os.path.join(path, "*.csv")
)

print("\nCSV files:")
print(csv_files)

# Step 4: Load dataset

df = pd.read_csv(csv_files[0])

print("\nFirst 5 rows:")
display(df.head())

# Step 5: Check dataset size

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Step 6: Check column names

print("\nColumn Names:")
print(df.columns)

# Step 7: Check missing values

print("\nMissing Values:")
print(df.isnull().sum())

if df.isnull().sum().sum() == 0:
    print("No missing values found.")
else:
    print("Missing values found.")

# Step 8: Check duplicate rows

print("\nDuplicate Rows:")
print(df.duplicated().sum())
# Remove duplicates if present
df = df.drop_duplicates()

# Step 9: Encode text data

# Convert Gender
if "GENDER" in df.columns:
    df["GENDER"] = df["GENDER"].map({
        "M": 1,
        "F": 0
    })

# Convert target variable
if "LUNG_CANCER" in df.columns:
    df["LUNG_CANCER"] = df["LUNG_CANCER"].map({
        "YES": 1,
        "NO": 0
    })

print("\nDataset after encoding:")
display(df.head())

# Step 10: Separate features and target

X = df.drop("LUNG_CANCER", axis=1)

y = df["LUNG_CANCER"]

print("X shape:", X.shape)
print("y shape:", y.shape)

# Step 11: Split training and testing data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)


# Step 12: Feature Selection

selector = SelectKBest(
    score_func=f_classif,
    k=10
)

X_train_selected = selector.fit_transform(
    X_train,
    y_train
)

X_test_selected = selector.transform(
    X_test
)


# Show selected feature names
selected_features = X.columns[
    selector.get_support()
]

print("\nSelected Features:")
print(selected_features)


# Step 13: Feature Scaling

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_selected
)

X_test_scaled = scaler.transform(
    X_test_selected
)

print("\n Watermark: ID: 2671507")
print("\nFeature scaling completed.")

## Phase B: Model Implementation

1. Trains Logistic Regression
2. Trains SVM
3. Generates predictions

In [ ]:
# Import models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# 1. Logistic Regression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model
logistic_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_logistic = logistic_model.predict(X_test_scaled)

print("Logistic Regression completed.")

# 2. Support Vector Machine (SVM)

svm_model = SVC(
    kernel="rbf",
    random_state=42
)

# Train the model
svm_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_svm = svm_model.predict(X_test_scaled)

print("SVM completed.")
# print("\n Watermark: ID: 2671507")
# Showing first 20 predictions for output check

print("\nLogistic Regression Predictions:")
print(y_pred_logistic[:20])

print("\nSVM Predictions:")
print(y_pred_svm[:20])

## Phase C: Evaluation & Visualization

This cell calculates and visualizes:
Accuracy, Precision, Recall, F1-score, Confusion Matrix, Model comparison, ROC curve and AUC


In [ ]:
# PHASE C: EVALUATION AND VISUALIZATION

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. LOGISTIC REGRESSION DETAILS

lr_accuracy = accuracy_score(y_test, y_pred_logistic)
lr_precision = precision_score(y_test, y_pred_logistic, zero_division=0)
lr_recall = recall_score(y_test, y_pred_logistic, zero_division=0)
lr_f1 = f1_score(y_test, y_pred_logistic, zero_division=0)

# print("\n Watermark: ID: 2671507")
print("LOGISTIC REGRESSION DETAILS")
print("----------------------------")
print("Accuracy :", round(lr_accuracy, 4))
print("Precision:", round(lr_precision, 4))
print("Recall   :", round(lr_recall, 4))
print("F1-score :", round(lr_f1, 4))

# Logistic Regression Confusion Matrix
lr_cm = confusion_matrix(
    y_test,
    y_pred_logistic,
    labels=[0, 1]
)
lr_tn, lr_fp, lr_fn, lr_tp = lr_cm.ravel()

print("\nConfusion Matrix Details")
print("----------------------------")
print("True Positive  (TP):", lr_tp)
print("True Negative  (TN):", lr_tn)
print("False Positive (FP):", lr_fp)
print("False Negative (FN):", lr_fn)

# Logistic Regression Heatmap
plt.figure(figsize=(6, 5))

sns.heatmap(
    lr_cm,
    annot=[
        [f"TN\n{lr_tn}", f"FP\n{lr_fp}"],
        [f"FN\n{lr_fn}", f"TP\n{lr_tp}"]
    ],
    fmt="",
    cmap="Blues",
    cbar=False,
    xticklabels=["No Cancer", "Cancer"],
    yticklabels=["No Cancer", "Cancer"]
)
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.show()

# 2. SVM DETAILS

svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_precision = precision_score(y_test, y_pred_svm, zero_division=0)
svm_recall = recall_score(y_test, y_pred_svm, zero_division=0)
svm_f1 = f1_score(y_test, y_pred_svm, zero_division=0)

print("SVM DETAILS")
print("----------------------------")
print("Accuracy :", round(svm_accuracy, 4))
print("Precision:", round(svm_precision, 4))
print("Recall   :", round(svm_recall, 4))
print("F1-score :", round(svm_f1, 4))

# SVM Confusion Matrix
svm_cm = confusion_matrix(
    y_test,
    y_pred_svm,
    labels=[0, 1]
)
svm_tn, svm_fp, svm_fn, svm_tp = svm_cm.ravel()

print("\nConfusion Matrix Details")
print("----------------------------")
print("True Positive  (TP):", svm_tp)
print("True Negative  (TN):", svm_tn)
print("False Positive (FP):", svm_fp)
print("False Negative (FN):", svm_fn)

# SVM Heatmap
plt.figure(figsize=(6, 5))

sns.heatmap(
    svm_cm,
    annot=[
        [f"TN\n{svm_tn}", f"FP\n{svm_fp}"],
        [f"FN\n{svm_fn}", f"TP\n{svm_tp}"]
    ],
    fmt="",
    cmap="Blues",
    cbar=False,
    xticklabels=["No Cancer", "Cancer"],
    yticklabels=["No Cancer", "Cancer"]
)
# print("\n ID: 26 715 07")
plt.title("SVM Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.show()

# 3. MODEL COMPARISON TABLE

comparison_table = pd.DataFrame({

    "Model": [
        "Logistic Regression",
        "SVM"
    ],

    "Accuracy": [
        lr_accuracy,
        svm_accuracy
    ],

    "Precision": [
        lr_precision,
        svm_precision
    ],

    "Recall": [
        lr_recall,
        svm_recall
    ],

    "F1-score": [
        lr_f1,
        svm_f1
    ]
})
print("MODEL COMPARISON")

display(comparison_table.round(4))

# 4. MODEL COMPARISON CHART

comparison_plot = comparison_table.melt(
    id_vars="Model",
    var_name="Metric",
    value_name="Score"
)
plt.figure(figsize=(10, 6))

sns.barplot(
    data=comparison_plot,
    x="Metric",
    y="Score",
    hue="Model"
)
plt.title("Logistic Regression vs SVM")
plt.xlabel("Evaluation Metric")
plt.ylabel("Score")

plt.ylim(0, 1)

plt.legend(title="Model")

plt.show()

# 5. ROC CURVE AND AUC

from sklearn.metrics import roc_curve, roc_auc_score

# Logistic Regression probability scores
lr_scores = logistic_model.predict_proba(X_test_scaled)[:, 1]

# SVM decision scores
svm_scores = svm_model.decision_function(X_test_scaled)

# Calculate AUC scores
lr_auc = roc_auc_score(y_test, lr_scores)
svm_auc = roc_auc_score(y_test, svm_scores)

# Calculate ROC curve values
lr_fpr, lr_tpr, _ = roc_curve(
    y_test,
    lr_scores
)

svm_fpr, svm_tpr, _ = roc_curve(
    y_test,
    svm_scores
)

# Print AUC results
print("ROC-AUC RESULTS")
print("----------------------------")

print("Logistic Regression AUC:", round(lr_auc, 4))
print("SVM AUC                :", round(svm_auc, 4))

# Plot ROC curves
plt.figure(figsize=(8, 6))

plt.plot(
    lr_fpr,
    lr_tpr,
    label=f"Logistic Regression (AUC = {lr_auc:.4f})"
)

plt.plot(
    svm_fpr,
    svm_tpr,
    label=f"SVM (AUC = {svm_auc:.4f})"
)

# Random classifier reference line
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.title("ROC Curve: Logistic Regression vs SVM")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.legend()

plt.grid()

plt.show()